In [1]:
# M4 - XGBoost + Target Encoding + Optuna Tuning
#
# ⚠️  DÉPENDANCES REQUISES :
# pip install category_encoders optuna xgboost
#
# 1. IMPORTATIONS POUR LE M3 (Sélection + XGBoost)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing classique
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

# Outils pour la sélection de variables et l'optimisation
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.exceptions import TrialPruned

# Le Modèle (Remplacement du Random Forest)
from xgboost import XGBRegressor

/home/awalther/anaconda3/envs/dataScience/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. CHARGEMENT DES DONNÉES ET PREPARATION DE LA TARGET

def advanced_feature_engineering(data):
    """
    Applique les transformations métiers sur le jeu de données.
    Note : L'encodage ordinal et les variables combinées (TotalSF, TotalBath, IsRemodeled)
    sont inspirés de stratégies performantes trouvées sur les notebooks publics de Kaggle.
    """
    df = data.copy()

    # --- A. CRÉATION DE VARIABLES (Inspiration Kaggle) ---
    # 1. Surface totale (Sous-sol + RDC + 1er étage)
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

    # 2. Salles de bain totales (Les "HalfBath" comptent pour 0.5)
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']

    # 3. Rénovation (Flag binaire : 1 si rénové, 0 sinon)
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)

    # --- B. ENCODAGE ORDINAL (Inspiration Kaggle) ---
    # On force la qualité en chiffres pour créer une hiérarchie stricte pour le Random Forest
    ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                    'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
    quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0, np.nan: 0}

    for col in ordinal_cols:
        if col in df.columns:
            df[col] = df[col].map(quality_map)

    # --- C. GESTION DES DATES (Calcul des âges) ---
    df['AgeBuilt'] = df['YrSold'] - df['YearBuilt']
    df['AgeRemodAdd'] = df['YrSold'] - df['YearRemodAdd']
    df['AgeGarage'] = df['YrSold'] - df['GarageYrBlt']

    # On supprime les anciennes dates brutes qui perturbent le modèle
    df = df.drop(['YearBuilt', 'YearRemodAdd', 'GarageYrBlt'], axis=1)

    return df


# --- 1. Chargement brut ---
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# --- 2. Nettoyage des Outliers (SUR LE TRAIN UNIQUEMENT) ---
# Suppression des 2 transactions aberrantes repérées visuellement (>4000 sqft & <300000$)
train_df = train_df.drop(train_df[train_df['GrLivArea'] > 4000].index)

# --- 3. Feature Engineering ---
train_df = advanced_feature_engineering(train_df)
test_df = advanced_feature_engineering(test_df)

# --- 4. Séparation X et y ---
X_train = train_df.drop(['Id', 'SalePrice'], axis=1)
y_train = pd.Series(np.log1p(train_df['SalePrice']), index=train_df.index)

X_test = test_df.drop(['Id'], axis=1)
test_ids = test_df['Id']

In [3]:
# 3. CRÉATION DU PREPROCESSOR (Optimisé : KNNImputer + Séparation des Catégories + Scaler)

# Redétection dynamique des colonnes après les changements de l'étape 2
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# 1. Identification des "faux manquants" (les variables où NaN = "Il n'y en a pas")
cat_none_features = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                     'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                     'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                     'BsmtFinType2', 'MasVnrType']

# Les autres catégorielles (les "vrais manquants")
cat_freq_features = [col for col in categorical_features if col not in cat_none_features]

# 2. Pipeline Numérique : Utilisation du KNNImputer
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', RobustScaler())
])

# 3. Pipeline Catégorielle 1 : Remplacement par "None"
cat_none_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('target_encoder', TargetEncoder(smoothing=10, handle_unknown='value', handle_missing='value')),
    ('scaler', RobustScaler()) # <--- AJOUT POUR PROTÉGER FACE AU LASSO
])

# 4. Pipeline Catégorielle 2 : Remplacement par le mode
cat_freq_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder(smoothing=10, handle_unknown='value', handle_missing='value')),
    ('scaler', RobustScaler()) # <--- AJOUT POUR PROTÉGER FACE AU LASSO
])

# 5. L'assembleur final (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat_none', cat_none_transformer, cat_none_features),
        ('cat_freq', cat_freq_transformer, cat_freq_features)
    ])

In [4]:
# 4. CONSTRUCTION DE LA PIPELINE HYBRIDE ET TUNING (M4 - XGBoost)

from sklearn.model_selection import cross_val_score

# Désactiver les logs d'Optuna pour ne pas polluer l'écran
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """
    Fonction d'objectif pour Optuna : Paramètres XGBoost calibrés SPÉCIFIQUEMENT
    pour contrer l'overfitting lié au TargetEncoder et au KNNImputer.
    """

    param = {
        # 1. Architecture de l'arbre (Moins profond pour éviter d'apprendre le TargetEncoder par coeur)
        'max_depth': trial.suggest_int('max_depth', 2, 6), # Baissé (ton collègue était à 3-8)
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 10), # Augmenté (plus conservateur)

        # 2. Vitesse d'apprentissage (Plus lente mais plus précise)
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),

        # 3. Échantillonnage (Prendre des bouts de données au hasard pour être robuste)
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8), # Plus bas pour ignorer les features inutiles

        # 4. Régularisation (Le "Lasso" et "Ridge" internes de XGBoost)
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),

        'random_state': 42,
        'n_jobs': -1
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # TA Pipeline Scikit-Learn (Preprocessor + XGBoost)
    pipeline_opt = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(**param))
    ])

    scores = cross_val_score(
        pipeline_opt,
        X_train,
        y_train,
        cv=kf,
        scoring='neg_root_mean_squared_error'
    )

    return -scores.mean()


# --- LANCEMENT DE L'OPTIMISATION ---
print("Lancement d'Optuna : Recherche des paramètres XGBoost (Optimisé TargetEncoder)...")
# On monte à 80 essais car on a un learning_rate plus petit et un espace plus complexe
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=80, show_progress_bar=True)

print("\n--- RÉSULTATS OPTIMISATION ---")
print(f"Meilleur RMSE (CV) : {study.best_value:.5f}")
print("Meilleurs Paramètres trouvés :")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


# --- ÉTAPE FINALE : RÉENTRAÎNER LE MEILLEUR MODÈLE ---
print("\nEntraînement du modèle final avec les meilleurs paramètres...")
best_params = study.best_params
xgb_best = XGBRegressor(**best_params, random_state=42, n_jobs=-1)

best_m4_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_best)
])

best_m4_pipeline.fit(X_train, y_train)

y_pred_train = best_m4_pipeline.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f"RMSE (Log) sur le Train Set : {rmse_train:.5f}")

Lancement d'Optuna : Recherche des paramètres XGBoost (Optimisé TargetEncoder)...


Best trial: 64. Best value: 0.113255: 100%|██████████| 80/80 [06:14<00:00,  4.68s/it]



--- RÉSULTATS OPTIMISATION ---
Meilleur RMSE (CV) : 0.11325
Meilleurs Paramètres trouvés :
  max_depth: 2
  min_child_weight: 4
  n_estimators: 1900
  learning_rate: 0.028815700631324635
  subsample: 0.6697381346575199
  colsample_bytree: 0.5255571901967997
  reg_alpha: 0.5216113279743188
  reg_lambda: 0.017942638185961902

Entraînement du modèle final avec les meilleurs paramètres...
RMSE (Log) sur le Train Set : 0.07579


In [5]:
# 5. ÉTAPE FINALE : SOUMISSION KAGGLE (M4 - XGBoost)

print("Génération des prédictions sur le jeu de test Kaggle...")

# Le modèle "best_m4_pipeline" est DÉJÀ entraîné à la fin de l'étape 4 !
# On l'utilise directement sur TES données de test (X_test)
log_predictions = best_m4_pipeline.predict(X_test)

# Transformation inverse (Expm1) pour revenir en Dollars ($)
final_predictions = np.expm1(log_predictions)

print("Création du fichier de soumission...")
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Sauvegarde au format CSV
fichier_soumission = 'submission_M4_XGBoost_Optuna.csv'
submission.to_csv(fichier_soumission, index=False)

print(f"\nC'est terminé ! Le fichier '{fichier_soumission}' est prêt à être soumis sur Kaggle.")
display(submission.head())

Génération des prédictions sur le jeu de test Kaggle...
Création du fichier de soumission...

C'est terminé ! Le fichier 'submission_M4_XGBoost_Optuna.csv' est prêt à être soumis sur Kaggle.


,Id,SalePrice
0,1461,123689.445312
1,1462,162110.000000
2,1463,185287.468750
3,1464,192542.703125
4,1465,192352.750000
